# Doanh thu mùa cao điểm tăng 5.6%: nhờ giá, nhờ lượng, nhờ mix, hay nhờ sản phẩm mới?

Bộ `online-retail-ii`, **cả hai sheet**: `Year 2009-2010` (01/12/2009 tới 09/12/2010) và `Year 2010-2011`
(01/12/2010 tới 09/12/2011). Hai sheet trùng nhau 01-09/12/2010, nên khi ghép chỉ lấy sheet đầu **trước**
01/12/2010. Quy tắc dòng hợp lệ dùng chung của track; doanh thu dòng = `Quantity * Price` (GBP, doanh thu gộp).

**Bối cảnh.** Giám đốc kinh doanh so mùa cao điểm (tháng 9 tới 11, ba tháng đủ ngày trước Giáng sinh) năm 2011
với năm 2010: doanh thu hàng hoá tăng. Câu hỏi của bà: "Mình tăng nhờ bán được nhiều hơn, nhờ giá cao hơn, hay nhờ
khách chuyển sang hàng đắt hơn? Có nên tăng giá tiếp không?" Bạn làm **price-volume-mix (PVM) bridge** ở cấp mã
hàng, tách riêng sản phẩm mới và sản phẩm ngừng bán.

Chỉ xét **hàng hoá**: `StockCode` bắt đầu bằng chữ số. Các mã như `POST`, `DOT`, `M`, `AMAZONFEE`, `BANK CHARGES`,
`gift_...` là phí, điều chỉnh hay voucher, không có "giá" và "lượng" theo nghĩa PVM.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/13-business-financial-analytics/exercises/intermediate/01-price-volume-mix -q`

In [ ]:
# explore: đọc hai sheet (lần đầu khoảng 15 giây, sau đó đọc cache), ghép và bỏ phần trùng
import os
import sys
import tempfile
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import duckdb
import pandas as pd

XLSX = ensure("online-retail-ii") / "online_retail_II.xlsx"


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet, đúng dữ liệu gốc. Lần đầu đọc xlsx (DuckDB khoảng 7 giây; máy không cài được extension excel
    thì pandas 30-90 giây), rồi cache parquet trong thư mục tạm (khoá theo mtime + size); lần sau khoảng 1 giây."""
    stat = XLSX.stat()
    cache = (Path(tempfile.gettempdir()) / "da-cache"
             / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet")
    if cache.exists():
        return pd.read_parquet(cache)
    path = str(XLSX).replace("'", "''")
    try:
        df = duckdb.sql(f"""
            SELECT Invoice, StockCode, Description, CAST(CAST(Quantity AS DOUBLE) AS BIGINT) AS Quantity,
                   TIMESTAMP '1899-12-30' + to_seconds(CAST(round(CAST(InvoiceDate AS DOUBLE) * 86400) AS BIGINT))
                       AS InvoiceDate,
                   CAST(Price AS DOUBLE) AS Price, CAST("Customer ID" AS DOUBLE) AS "Customer ID", Country
            FROM read_xlsx('{path}', sheet = '{sheet}', all_varchar = true)""").df()
    except (duckdb.CatalogException, duckdb.IOException, duckdb.HTTPException) as exc:
        if "extension" not in str(exc).lower():
            raise
        df = pd.read_excel(XLSX, sheet_name=sheet)          # máy offline: không cài được extension excel
    df = df.assign(Invoice=df["Invoice"].astype(str), StockCode=df["StockCode"].astype(str),
                   Description=df["Description"].astype("string"))
    cache.parent.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{os.getpid()}.tmp")
    df.to_parquet(tmp, index=False)
    os.replace(tmp, cache)
    return df

first, second = read_sheet("Year 2009-2010"), read_sheet("Year 2010-2011")
tx = pd.concat([first[first.InvoiceDate < "2010-12-01"], second], ignore_index=True)
print(len(first), "+", len(second), "→", len(tx), "dòng sau khi bỏ trùng 01-09/12/2010")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Doanh thu và số lượng theo mã hàng

`product_sales(tx, start, end) -> pd.DataFrame`: trên các dòng hợp lệ là hàng hoá (`StockCode` ép `str`, bắt đầu
bằng chữ số) có `InvoiceDate` trong `[start, end)`, mỗi `StockCode` một dòng (index tên `StockCode`, tăng dần),
cột theo thứ tự:

- `revenue`: tổng `Quantity * Price`;
- `units`: tổng `Quantity`;
- `price`: `revenue / units`, **giá bình quân thực thu** (một mã có nhiều mức giá theo số lượng mua sỉ).

In [ ]:
def product_sales(tx: pd.DataFrame, start: str, end: str) -> pd.DataFrame:
    """Hàng hoá hợp lệ trong [start, end): revenue, units, price theo StockCode."""
    invoice, code = tx["Invoice"].astype(str), tx["StockCode"].astype(str)
    keep = (~invoice.str.startswith("C") & (tx["Quantity"] > 0) & (tx["Price"] > 0)
            & code.str.match(r"\d") & (tx["InvoiceDate"] >= start) & (tx["InvoiceDate"] < end))
    rows = pd.DataFrame({"StockCode": code[keep], "revenue": tx.loc[keep, "Quantity"] * tx.loc[keep, "Price"],
                         "units": tx.loc[keep, "Quantity"]})
    out = rows.groupby("StockCode")[["revenue", "units"]].sum().sort_index()
    out["price"] = out["revenue"] / out["units"]
    return out

## 2. Price-volume-mix bridge

Gọi kỳ gốc là 0, kỳ so sánh là 1. Sản phẩm **chung** là mã có bán ở cả hai kỳ. Trên các sản phẩm chung, với giá
p và số lượng q của từng mã, tổng số lượng Q, tỉ trọng số lượng s = q / Q:

$$
\text{price} = \sum_i (p_{1i} - p_{0i})\, q_{1i}, \qquad
\text{volume} = (Q_1 - Q_0) \sum_i s_{0i}\, p_{0i}, \qquad
\text{mix} = Q_1 \sum_i (s_{1i} - s_{0i})\, p_{0i}
$$

- `volume`: nếu chỉ tổng số lượng đổi, mix và giá giữ nguyên kỳ gốc;
- `mix`: khách chuyển sang mã đắt hơn (dương) hay rẻ hơn (âm), tính bằng giá kỳ gốc;
- `price`: giá từng mã đổi, tính trên số lượng kỳ mới.

Ba hiệu ứng cộng lại đúng bằng thay đổi doanh thu của sản phẩm chung. Thêm hai dòng:

- `new` = doanh thu kỳ 1 của mã chỉ có ở kỳ 1;
- `lost` = **trừ** doanh thu kỳ 0 của mã chỉ có ở kỳ 0.

`pvm_bridge(p0, p1) -> pd.Series`: index `price`, `volume`, `mix`, `new`, `lost` theo thứ tự, tên `effect`; tổng
năm dòng bằng `p1.revenue.sum() - p0.revenue.sum()`. `p0`, `p1` là kết quả của `product_sales`.

`product_effects(p0, p1) -> pd.DataFrame`: chỉ sản phẩm chung, index `StockCode` tăng dần, cột `revenue0`,
`revenue1`, `price_effect` = (p1 − p0) · q1, `quantity_effect` = p0 · (q1 − q0). Tổng `price_effect` bằng
`price` của bridge, tổng `quantity_effect` bằng `volume + mix`.

In [ ]:
def pvm_bridge(p0: pd.DataFrame, p1: pd.DataFrame) -> pd.Series:
    """price, volume, mix (sản phẩm chung), new, lost; tên "effect"."""
    common = p0.index.intersection(p1.index)
    c0, c1 = p0.loc[common], p1.loc[common]
    q0, q1 = c0["units"].sum(), c1["units"].sum()
    s0, s1 = c0["units"] / q0, c1["units"] / q1
    return pd.Series({
        "price": ((c1["price"] - c0["price"]) * c1["units"]).sum(),
        "volume": (q1 - q0) * (s0 * c0["price"]).sum(),
        "mix": q1 * ((s1 - s0) * c0["price"]).sum(),
        "new": p1.drop(common)["revenue"].sum(),
        "lost": -p0.drop(common)["revenue"].sum(),
    }, name="effect")


def product_effects(p0: pd.DataFrame, p1: pd.DataFrame) -> pd.DataFrame:
    """Sản phẩm chung: revenue0, revenue1, price_effect, quantity_effect."""
    common = p0.index.intersection(p1.index).sort_values()
    c0, c1 = p0.loc[common], p1.loc[common]
    out = pd.DataFrame({
        "revenue0": c0["revenue"], "revenue1": c1["revenue"],
        "price_effect": (c1["price"] - c0["price"]) * c1["units"],
        "quantity_effect": c0["price"] * (c1["units"] - c0["units"]),
    })
    out.index.name = "StockCode"
    return out

In [ ]:
# explore: bridge mùa cao điểm 09-11/2010 → 09-11/2011
p0 = product_sales(tx, "2010-09-01", "2010-12-01")
p1 = product_sales(tx, "2011-09-01", "2011-12-01")
bridge = pvm_bridge(p0, p1)
r0, r1 = p0.revenue.sum(), p1.revenue.sum()
print(f"doanh thu hàng hoá: {r0:,.0f} → {r1:,.0f} GBP ({r1 / r0 - 1:+.1%}); kiểm tổng bridge: {bridge.sum():,.0f}")
print(bridge.round(0).to_string())
common = p0.index.intersection(p1.index)
print(f"mã: kỳ 0 {len(p0)}, kỳ 1 {len(p1)}, chung {len(common)}, mới {len(p1) - len(common)}, "
      f"ngừng {len(p0) - len(common)}")
print(f"số lượng sản phẩm chung: {p0.loc[common, 'units'].sum():,} → {p1.loc[common, 'units'].sum():,}")

In [ ]:
# explore: waterfall cho Giám đốc kinh doanh
labels = ["09-11/2010", *bridge.index, "09-11/2011"]
level = r0 + bridge.cumsum().shift(fill_value=0)
heights = [r0, *bridge.abs(), r1]
bottoms = [0, *np.minimum(level, level + bridge), 0]
colors = ["#9aa5b1", *["#4c78a8" if v >= 0 else "#e45756" for v in bridge], "#9aa5b1"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, np.array(heights) / 1e6, bottom=np.array(bottoms) / 1e6, color=colors)
ax.set_ylabel("triệu GBP")
ax.set_ylim(2.5, None)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Tăng trưởng đến từ mã mới; mã cũ bán ít đi", loc="left")
plt.show()

In [ ]:
# explore: mã nào kéo hiệu ứng giá xuống? Giá thực thu giảm vì giảm giá hay vì bán sỉ nhiều hơn?
eff = product_effects(p0, p1)
named = tx.assign(StockCode=tx.StockCode.astype(str)).dropna(subset=["Description"])
desc = named.groupby(["StockCode", "Description"]).size().sort_values().groupby(level=0).tail(1)
desc = desc.reset_index(level=1).Description          # mô tả hay gặp nhất của mỗi mã
view = eff.join(p0[["price", "units"]].add_suffix("0")).join(p1[["price", "units"]].add_suffix("1")).join(desc)
print(view.nsmallest(6, "price_effect").round(2).to_string())
lines = tx[tx.StockCode.astype(str).isin(view.nsmallest(6, "price_effect").index) & (tx.Quantity > 0)
           & (tx.Price > 0) & ~tx.Invoice.astype(str).str.startswith("C")]
lines = lines.assign(period=np.where(lines.InvoiceDate.between("2010-09-01", "2010-11-30 23:59"), "2010",
                     np.where(lines.InvoiceDate.between("2011-09-01", "2011-11-30 23:59"), "2011", None))).dropna(subset=["period"])
print(lines.groupby(["StockCode", "period"]).agg(min_price=("Price", "min"), median_price=("Price", "median"),
                                                 max_price=("Price", "max"), lines=("Price", "size"),
                                                 median_qty=("Quantity", "median")).round(2))

In [ ]:
# explore: mã mới lớn nhất và phần doanh thu không phải hàng hoá
print(p1.drop(common).join(desc).nlargest(8, "revenue").round(2))
valid = tx[~tx.Invoice.astype(str).str.startswith("C") & (tx.Quantity > 0) & (tx.Price > 0)]
valid = valid.assign(revenue=valid.Quantity * valid.Price)
for a, b in [("2010-09-01", "2010-12-01"), ("2011-09-01", "2011-12-01")]:
    part = valid[(valid.InvoiceDate >= a) & (valid.InvoiceDate < b)]
    print(a[:7], f"doanh thu gộp {part.revenue.sum():,.0f}, trong đó không phải hàng hoá "
          f"{part[~part.StockCode.astype(str).str.match(r'\d')].revenue.sum():,.0f}")

## 3. Đọc kết quả, rồi làm trắc nghiệm

1. Viết cho Giám đốc kinh doanh một câu: doanh thu tăng bao nhiêu, phần lớn nhất đến từ đâu, phần nào âm.
2. Hiệu ứng giá âm: mã nào kéo xuống nhiều nhất? Bảng min/median/max giá của chúng cho thấy giảm giá niêm yết
   hay đổi cơ cấu mức giá sỉ? Trả lời câu "có nên tăng giá tiếp không" thế nào cho trung thực?
3. Số lượng của sản phẩm chung giảm khoảng bao nhiêu phần trăm? Có phải "khách mua ít đi", hay là danh mục được
   thay mới (mã mới thay mã cũ cùng loại)? Cần dữ liệu gì để phân biệt?
4. Đổi thứ tự định nghĩa (tính `price` trên số lượng kỳ gốc, `volume` theo giá kỳ mới) thì các hiệu ứng đổi bao
   nhiêu? Vì sao báo cáo phải ghi rõ công thức?

Xong thì mở app, làm phần **trắc nghiệm** của bài.